# Data Preparation: Train/Val/Test Splits
Creates balanced 60/20/20 splits from human and AI reviews

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Load datasets
human_df = pd.read_csv('tripadvisor_hotel_reviews.csv')
ai_df = pd.read_csv('ai_generated_tripadvisor_reviews_gemma3_4b.csv')

print(f"Human reviews: {len(human_df)}")
print(f"AI reviews: {len(ai_df)}")

In [ ]:
# Add labels
human_df['label'] = 0  # Human
ai_df['label'] = 1     # AI-generated

# Undersample human reviews to match AI reviews (10,000 each)
human_df_sampled = human_df.sample(n=len(ai_df), random_state=42)

print(f"Balanced dataset: {len(human_df_sampled)} human + {len(ai_df)} AI = {len(human_df_sampled) + len(ai_df)} total")

In [ ]:
# Combine and shuffle
combined_df = pd.concat([human_df_sampled, ai_df], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Combined dataset: {len(combined_df)} reviews")
print(f"Label distribution:\n{combined_df['label'].value_counts()}")

In [ ]:
# Split: 60% train, 20% val, 20% test
train_df, temp_df = train_test_split(combined_df, test_size=0.4, random_state=42, stratify=combined_df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"\nTrain: {len(train_df)} ({len(train_df)/len(combined_df)*100:.1f}%)")
print(f"Val:   {len(val_df)} ({len(val_df)/len(combined_df)*100:.1f}%)")
print(f"Test:  {len(test_df)} ({len(test_df)/len(combined_df)*100:.1f}%)")

In [ ]:
# Save splits
train_df.to_csv('train_split.csv', index=False)
val_df.to_csv('val_split.csv', index=False)
test_df.to_csv('test_split.csv', index=False)

print("\n✓ Splits saved!")

In [ ]:
# Verify label balance in each split
print("\nLabel distribution in splits:")
print("Train:", train_df['label'].value_counts().to_dict())
print("Val:  ", val_df['label'].value_counts().to_dict())
print("Test: ", test_df['label'].value_counts().to_dict())